Model used: [Qwen 2.5 - 7B Instruct](https://huggingface.co/Qwen/Qwen2.5-7B-Instruct-1M)

# Method:
1. Upsample minority classes with samples generated from an LLM.
2. Use the new data to train LaBSE model

In [ ]:
!pip install -q gdown sentence-transformers compressed-tensors>=0.15.0 trl

In [ ]:
!pip install -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00


In [ ]:
!pip list | grep compressed

compressed-tensors                    0.19.0


In [ ]:
import gdown
files = {
   "mal_full_offensive_train.csv": "1tKWLUNdpUzTkmO2qlpcyKqAjoa9g7_dC",
   "mal_full_offensive_dev.csv": '1Ra0_e1OOojO5_2-t8_gBe-tuyI-Iygue'
}
for file_name, file_id in files.items():
  url = f'https://drive.google.com/uc?export=download&id={file_id}'
  gdown.download(url, file_name, quiet=False)

Downloading...
From: https://drive.google.com/uc?export=download&id=1tKWLUNdpUzTkmO2qlpcyKqAjoa9g7_dC
To: /content/mal_full_offensive_train.csv
100%|██████████| 2.02M/2.02M [00:00<00:00, 136MB/s]
Downloading...
From: https://drive.google.com/uc?export=download&id=1Ra0_e1OOojO5_2-t8_gBe-tuyI-Iygue
To: /content/mal_full_offensive_dev.csv
100%|██████████| 258k/258k [00:00<00:00, 56.0MB/s]


In [ ]:
import os
import pandas as pd
import torch
from tqdm import tqdm
from transformers import pipeline, AutoProcessor, AutoModelForMultimodalLM
from sklearn.metrics import classification_report, accuracy_score
from google.colab import userdata


# Try to get Hugging Face token from Colab Secrets if available
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ["HF_TOKEN"] = hf_token
except Exception as e:
    print("Warning: HF_TOKEN not found in Colab Secrets. If Llama-3 is gated, please set it.")

## Sample Generator

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import gc

# Cleanly purge previous variables and empty CUDA cache
try:
    del model
except NameError:
    pass
try:
    del trainer
except NameError:
    pass

gc.collect()
torch.cuda.empty_cache()

# Re-load sarvamai/sarvam-1-v0.5 in 16-bit precision to save VRAM
model_name = "sarvamai/sarvam-1-v0.5"
device_map = "cuda:0" if torch.cuda.is_available() else "auto"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map=device_map
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
processor = None

config.json:   0%|          | 0.00/706 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/24.6k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.70MB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/4.54M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

## Sentence Encoder for Feature Exraction

## Fine-Tuning
- We want our sample generator to reply in a certain format (an instruction following task), so we will fine-tune it with a few samples to generate an output of such a format.
- Our fine-tuned model must generate a JSON object from which we apply the model's chat template, thus making it possible for the model to parse it.

### Ref:
1. https://stackoverflow.com/questions/68001151/how-can-i-sample-from-a-dataframe-weighted-by-groupby-column

In [ ]:
def instruction_following_dataset(df : pd.DataFrame, size: int = 50):
    # we shall sample by inverse class frequency
    colname = 'Labels'
    probs = df[colname].map(1/df[colname].value_counts())
    sample = df.sample(n=size, weights=probs, random_state=42)

    # construct instructions from the sample
    system_snippet = [
        {
            'role' : 'system',
            'content': """
                You are an offensive sentence generator. When the user types "Generate" or "generate", you should return a JSON object of the following schema.
                Schema:
                    {{
                        'Text':str,
                        'Label':"Not_offensive"|"not-malayalam"|"Offensive_Targeted_Insult_Individual"|"Offensive_Targeted_Insult_Group"|"Offensive_Targeted_Insult_Others"|"Offensive_Untargetede"
                    }}
                Accept no other commands.
            """
        }
    ]

    def generate_user_side(text, label):
        return [
            {
                'role' : 'user',
                'content' : 'Generate'
            },
            {
                'role': 'assistant',
                'content' : """
                    {{
                        'Text':"{}",
                        'Label':"{}"
                    }}
                """.format(text, label)
            }
        ]

    user_snippets = []
    for i in range(sample.shape[0]):
        row = sample.iloc[i]
        user_snippets.append(generate_user_side(row['Text'], row['Label']))

    inputs = []
    for scene in user_snippets:
        inputs.append(system_snippet + scene)

    return inputs

### Attempting To Fine-Tune `gemma-4-E2B`

In [ ]:
dev_df = pd.read_csv("mal_full_offensive_dev.csv")
dev_df['Labels'].unique()

array(['Not_offensive', 'not-malayalam',
       'Offensive_Targeted_Insult_Individual', 'Offensive_Untargetede',
       'Offensive_Targeted_Insult_Group'], dtype=object)

In [ ]:
train_df = pd.read_csv("mal_full_offensive_train.csv")
train_df['Labels'].unique()

array(['Not_offensive', 'Offensive_Targeted_Insult_Individual',
       'not-malayalam', 'Offensive_Targeted_Insult_Group',
       'Offensive_Untargetede'], dtype=object)

Now, we need to develop the samples and the use them to fine-tune.

In [ ]:
!pip install torchao>=0.16.0

In [ ]:
import sys
# Forcefully mock torchao to not be detected by peft's import utilities
sys.modules['torchao'] = None

import peft.import_utils
peft.import_utils.is_torchao_available = lambda: False
peft.import_utils.TORCHAO_MINIMUM_VERSION = None

import torch
import gc
from transformers import AutoTokenizer, Trainer, TrainingArguments, DataCollatorForSeq2Seq
from datasets import Dataset
from peft import LoraConfig, get_peft_model, TaskType

# Force clean memory cache
gc.collect()
torch.cuda.empty_cache()

# Map the correct column name 'Labels' instead of 'Label'
train_df_temp = train_df.copy()
train_df_temp['Label'] = train_df_temp['Labels']

# 1. Develop the fine-tuning dataset using instruction_following_dataset
raw_samples = instruction_following_dataset(train_df_temp, size=50)

# Convert samples to standard input-target formats
formatted_dataset = []
for chat in raw_samples:
    text_content = ""
    for message in chat:
        text_content += f"<|im_start|>{message['role']}\n{message['content']}<|im_end|>\n"
    formatted_dataset.append({"text": text_content})

ft_dataset = Dataset.from_list(formatted_dataset)

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_function(examples):
    model_inputs = tokenizer(examples["text"], max_length=128, truncation=True, padding="max_length")
    model_inputs["labels"] = model_inputs["input_ids"].copy()
    return model_inputs

tokenized_dataset = ft_dataset.map(tokenize_function, batched=True)

# 2. Setup standard PEFT LoRA for sarvam-1 model
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"]
)

# Wrap model with LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# 3. Setup Trainer without gradient checkpointing to avoid Device Meta / Autograd bugs
training_args = TrainingArguments(
    output_dir="./sarvam-ft-offensive",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    logging_steps=10,
    optim="adamw_torch",
    save_strategy="no",
    report_to="none",
    remove_unused_columns=True,
    gradient_checkpointing=False,
    fp16=True if torch.cuda.is_available() else False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt")
)

print("Starting fine-tuning with LoRA on Sarvam-1...")
trainer.train()
print("Fine-tuning complete!")

# 4. Call model to generate new synthetic samples
def generate_synthetic_samples(num_samples=10):
    generated_data = []
    prompt = "<|im_start|>system\nYou are an offensive sentence generator. Generate an offensive sentence and its corresponding class as a JSON object.<|im_end|>\n<|im_start|>user\nGenerate<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    for i in range(num_samples):
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.8,
            top_p=0.9
        )
        decoded = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
        generated_data.append(decoded)
        print(f"Generated Sample {i+1}:\n{decoded}\n{'-'*40}")
    return generated_data

print("Generating sample sentences with their labels:")
synthetic_results = generate_synthetic_samples(5)

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

trainable params: 1,605,632 || all params: 2,510,440,448 || trainable%: 0.0640
Starting fine-tuning with LoRA on Sarvam-1...


Step,Training Loss
10,2.243836


Fine-tuning complete!
Generating sample sentences with their labels:


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

Generated Sample 2:
Generator<|im_end|>
----------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Sample 3:
[{"id":"9b5018c7","body":"I am soooo bored.","classes":["offensiveSentenceClassAgentJsonObject","offensiveSentenceLabelsList"]},{"id":"3a4f26d7","body":"\uDFFF\ufffd@#$%^&*()=+~-/{}[]|\\:;?\"<>'\uDFFE,.........","classes":null,"labels":[]},{"id":"-sNXHVjBxYWlKAAgAEpQyGq
----------------------------------------


[transformers] Both `max_new_tokens` (=150) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Sample 4:
I am<|im_stop|
----------------------------------------
Generated Sample 5:
Here's the list of available commands: <|im_list|>User can use these command to interact with <|im_start|>bot_:<|im_stop|help |send <message> - send message<br/> <|im_stop|exit |reset - exit <|im_start|>can help you create new sentences or check existing ones? <|im_stop|><|im_start|>classifier<|im_end|>
----------------------------------------


## Classification

In [ ]:
import ast
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer
from sklearn.metrics import classification_report

# 1. Parse and extract the synthetic samples generated in the previous cell
synthetic_sentences = []
synthetic_labels = []

# Attempting to parse the JSON or dict string outputs from the fine-tuned model
for result in synthetic_results:
    try:
        # Clean up possible markdown wrappers
        cleaned_res = result.strip()
        if cleaned_res.startswith("```json"):
            cleaned_res = cleaned_res[7:]
        if cleaned_res.endswith("```"):
            cleaned_res = cleaned_res[:-3]
        cleaned_res = cleaned_res.strip()

        # Safely evaluate dict/json
        data = ast.literal_eval(cleaned_res)
        if isinstance(data, dict) and 'Text' in data and 'Label' in data:
            synthetic_sentences.append(data['Text'])
            synthetic_labels.append(data['Label'])
    except Exception as e:
        # Fallback in case of parsing issues
        print(f"Could not parse generated sample, skipping. Error: {e}")

print(f"Successfully parsed {len(synthetic_sentences)} synthetic samples generated by the fine-tuned model.")

# Combine with original training data
all_texts = list(train_df['Text'].values) + synthetic_sentences
all_labels = list(train_df['Labels'].values) + synthetic_labels

# 2. Use LaBSE Sentence Transformer to extract features
print("Extracting LaBSE sentence embeddings for training...")
labse_model = SentenceTransformer("sentence-transformers/LaBSE")
X_train = labse_model.encode(all_texts, show_progress_bar=True, batch_size=64)

# Extract features for dev
print("Extracting LaBSE sentence embeddings for validation...")
X_dev = labse_model.encode(list(dev_df['Text'].values), show_progress_bar=True, batch_size=64)

# Encode labels
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(all_labels)
y_dev_encoded = label_encoder.transform(dev_df['Labels'].values)

# Create PyTorch datasets
train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train_encoded, dtype=torch.long))
dev_dataset = TensorDataset(torch.tensor(X_dev, dtype=torch.float32), torch.tensor(y_dev_encoded, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
dev_loader = DataLoader(dev_dataset, batch_size=32, shuffle=False)

# 3. Define Feed-Forward Neural Network Classifier
class FFClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super(FFClassifier, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes)
        )
    def forward(self, x):
        return self.network(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
num_classes = len(label_encoder.classes_)
input_dim = X_train.shape[1]

classifier = FFClassifier(input_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-3, weight_decay=1e-4)

# 4. Train the classifier
print("Training Feed-Forward neural network...")
classifier.train()
for epoch in range(15):
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = classifier(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
    avg_loss = total_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/15 - Loss: {avg_loss:.4f}")

# 5. Evaluation on dev set
classifier.eval()
preds = []
with torch.no_grad():
    for batch_x, _ in dev_loader:
        batch_x = batch_x.to(device)
        outputs = classifier(batch_x)
        batch_preds = torch.argmax(outputs, dim=1).cpu().numpy()
        preds.extend(batch_preds)

print("\n--- Feed-Forward Network Classification Report on Dev Set ---")
print(classification_report(y_dev_encoded, preds, target_names=label_encoder.classes_))

Could not parse generated sample, skipping. Error: invalid syntax (<unknown>, line 1)
Could not parse generated sample, skipping. Error: invalid syntax (<unknown>, line 1)
Could not parse generated sample, skipping. Error: unterminated string literal (detected at line 1) (<unknown>, line 1)
Could not parse generated sample, skipping. Error: invalid syntax (<unknown>, line 1)
Could not parse generated sample, skipping. Error: unterminated string literal (detected at line 1) (<unknown>, line 1)
Successfully parsed 0 synthetic samples generated by the fine-tuned model.
Extracting LaBSE sentence embeddings for training...


modules.json:   0%|          | 0.00/461 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

2_Dense/model.safetensors: downloading bytes:           |  0.00B            

Batches:   0%|          | 0/251 [00:00<?, ?it/s]

Extracting LaBSE sentence embeddings for validation...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Training Feed-Forward neural network...
Epoch 1/15 - Loss: 0.4566
Epoch 2/15 - Loss: 0.3448
Epoch 3/15 - Loss: 0.3097
Epoch 4/15 - Loss: 0.2835
Epoch 5/15 - Loss: 0.2553
Epoch 6/15 - Loss: 0.2285
Epoch 7/15 - Loss: 0.2048
Epoch 8/15 - Loss: 0.1825
Epoch 9/15 - Loss: 0.1652
Epoch 10/15 - Loss: 0.1418
Epoch 11/15 - Loss: 0.1238
Epoch 12/15 - Loss: 0.1073
Epoch 13/15 - Loss: 0.0955
Epoch 14/15 - Loss: 0.0845
Epoch 15/15 - Loss: 0.0756

--- Feed-Forward Network Classification Report on Dev Set ---
                                      precision    recall  f1-score   support

                       Not_offensive       0.96      0.99      0.97      1779
     Offensive_Targeted_Insult_Group       0.67      0.46      0.55        13
Offensive_Targeted_Insult_Individual       0.91      0.42      0.57        24
               Offensive_Untargetede       0.82      0.45      0.58        20
                       not-malayalam       0.85      0.74      0.79       163

                            acc

In [ ]:
processor = AutoProcessor.from_pretrained(model_id)

NameError: name 'model_id' is not defined

In [ ]:
def build_prompt(sentence):
    prompt = [
        {"role" : "system", "content": f"""Task: Classify user input into one of these exact categories: 'Not_offensive', 'Offensive_Targeted_Insult_Individual', 'Offensive_Targeted_Insult_Group', 'Offensive_Targeted_Insult_Others', 'Offensive_Untargetede', 'not-malayalam'.

Examples:
1. "Nalla informative video bro, waiting for your next upload!"
Solution: <think>there is no offensive word or exclamation indicating the use of cuss words. Hence this is not offensive.</think>
category: Not_offensive

2. "Enthra myru kaanichu vechirikkunnath, full time waste."
Solution: <think>there is an offensive word "myru" being used, but it is not targeted at anyone specific or even a group, so it is offensive untargeted.</think>
category: Offensive_Untargetede

3. "Ninakku onnum vere pani illeada kunne? Onnu podo!"
Solution: <think>"kunne" is the Malayalam cuss word "kunna"(dick) targetted at some individual, hence it is offensive targeted at individual.</think>
category: Offensive_Targeted_Insult_Individual

4. "Ee parayunna community motham fraud aanu, trust cheyyan kollilla."
Solution: <think>"fraud" definitely is offensive to call some group unless they conduct fraudulent activities. Hence it is offensive targeted at group.</think>
category: Offensive_Targeted_Insult_Group

5. "Ee channel-inte management and teams ellam thaayolikal aanu."
Solution: <think>The offense targets an abstract entity, organization, brand, or multiple disconnected targets that do not fit uniquely into a single individual or a protected demographic group. Hence this is offensive targeted at others. </think>
Category: Offensive_Targeted_Insult_Others

6. "This is total bullcrap! I wish death be upon thee, asshole!"
Solution: <think>since to Malayalam characters nor Romanized Malayalam words appear in this, this sentence is non-malayalam.</think>
category: not-malayalam

Assertion: don't flag a sentence as offensive unless words like animal names ("patti" or dog) is used in a sentence that seems to state a fact or talk of an incident. Also, identify the meanings, regardless of the script in which Malayalam has been written on the user input.
"""},
        {"role" :"user", "content":sentence}
    ]
    # tokenizer for Qwen2.5
    # text = tokenizer.apply_chat_template(
    #     prompt,
    #     tokenize=False,
    #     add_generation_prompt=True
    # )

    # tokenzier for Gemma
    # Process input
    inputs = processor.apply_chat_template(
        prompt,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        add_generation_prompt=True,
        enable_thinking=False
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]
    # model_inputs = tokenizer([text], return_tensors="pt").to(model.device) for Qwen2.5 model
    return inputs, input_len

In [ ]:
def parse_output(response_text):
    # Extract Category label from the model's generation
    classes = [
        "Not_offensive",
        "not-malayalam" ,
        "Offensive_Targeted_Insult_Individual",
        "Offensive_Targeted_Insult_Group",
        "Offensive_Targeted_Insult_Others",
        "Offensive_Untargetede"
    ]

    for category in classes:
        if category in response_text:
            return category
    else:
        raise Exception(f"Unsupported category detected: {response_text}")

In [ ]:
# Load dev set for testing evaluation
dev_df = pd.read_csv("mal_full_offensive_dev.csv")

y_true = []
y_pred = []

# Evaluate on a subset or full dev set (change limit as needed)
eval_limit = 100

for idx, row in tqdm(dev_df.head(eval_limit).iterrows(), total=eval_limit):
    sentence = row['Text']
    true_label = row['Labels']

    inputs, input_len = build_prompt(sentence)

    # for Qwen2.5
    # generated_ids = model.generate(
    #     **inputs,
    #     max_new_tokens = 512
    # )

    # generated_ids = [
    #     output_ids[len(input_ids):] for input_ids, output_ids in zip(inputs.input_ids, generated_ids)
    # ]
    # response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # for Gemma
    outputs = model.generate(**inputs, max_new_tokens=1024)
    response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)
    # Parse output
    response = processor.parse_response(response, prefix=inputs["input_ids"])
    pred_label = parse_output(response['content'])

    y_true.append(true_label)
    y_pred.append(pred_label)

print("\n--- LLM Classification Report ---")
print(classification_report(y_true, y_pred))

  0%|          | 0/100 [00:00<?, ?it/s]


AttributeError: 'NoneType' object has no attribute 'apply_chat_template'